# Day 2 – Prompts, Memory and Python for AI Agents

**Workshop:** One Person Company – AI Agents Workshop  
**Day:** 2  
**Project:** Sweet Crumbs Home Bakery Chatbot  
**Tools:** Python, Google Colab, Groq API, OpenAI Python SDK

## Learning Objectives

Today I learned how to:

1. Understand messages and roles in an LLM conversation.
2. Design structured system prompts using ROLE, FACTS, RULES and FORMAT.
3. Use few-shot prompting to guide model responses.
4. Generate structured JSON responses and parse them using Python.
5. Work with Python dictionaries and JSON.
6. Create functions using type hints and docstrings.
7. Understand decorators and use `@log_call`.
8. Handle API errors using `try` and `except`.
9. Maintain conversation history as chatbot memory.
10. Trim old conversation messages to manage memory.
11. Combine these concepts into a working chatbot.

**Today's goal:** Upgrade the Sweet Crumbs Home Bakery chatbot so that it can respond using structured instructions, handle information safely and remember previous messages.

In [1]:
!pip install openai

## 1. API Setup – Connecting Python to Groq

I am using Groq's OpenAI-compatible API through the OpenAI Python SDK.

This allows Python to send messages to the language model and receive responses.

The API key is retrieved securely from Google Colab Secrets rather than being written directly in the notebook.

**Model:** `openai/gpt-oss-20b`

In [2]:
from openai import OpenAI
from google.colab import userdata
import os

os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')

client = OpenAI(
    api_key=os.environ["GROQ_API_KEY"],
    base_url="https://api.groq.com/openai/v1"
)
print("Groq connection setup completed!")

Groq connection setup completed!


In [3]:
response = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[
        {"role": "user", "content": "What is an AI agent? Explain in 2 simple sentences."}
    ]
)
print(response.choices[0].message.content)

An AI agent is a software system that perceives its environment through sensors and acts on that environment to achieve goals. It uses algorithms and data to make decisions, learn, and improve its behavior over time.


### Observation — API Setup

- Installed the OpenAI Python package.
- Connected to Groq using its OpenAI-compatible API endpoint.
- Retrieved the API key securely from Google Colab Secrets.
- Successfully tested the `openai/gpt-oss-20b` model.

**Learning:** An OpenAI-compatible API allows us to use the OpenAI Python client with a compatible provider such as Groq by changing the API endpoint.

## 1. Messages and Roles

An LLM conversation uses messages with different roles.

- `system`: Gives instructions to the AI.
- `user`: Contains the user's message.
- `assistant`: Contains the AI's response.

**Learning:** Roles help the model understand who is speaking and what instructions it should follow.

In [4]:
messages = [
    {"role": "system", "content": "You are a friendly bakery assistant."},
    {"role": "user", "content": "What cakes do you recommend?"}
]

response = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=messages
)

print(response.choices[0].message.content)

🥰 **Hey there, cake lover!**  
Let’s find the perfect sweet treat for you. Here are a few of our top‑rated cakes (and a quick note about any dietary tweaks you might need):

| Cake | What makes it special | Ideal for |
|------|-----------------------|-----------|
| **Classic Rich Chocolate** | Double‑layer chocolate sponge, silky ganache, chocolate shavings on top | Anyone who wants pure, indulgent chocolate |
| **Velvet & Cream** | Moist red‑velvet cake with a velvety cream cheese frosting | Perfect for birthdays or celebrations |
| **Lemon Zest Delight** | Light citrus cake with a bright lemon‑y icing and candied zest | Great for spring, or if you want something refreshing |
| **Carrot‑Cinnamon Heaven** | Sweet carrot cake with warm spices, topped with cream cheese frosting | Ideal for a cozy, comforting bite |
| **Vanilla Bean Dream** | Light vanilla sponge, vanilla buttercream, fresh berries | Classic, elegant, and always a crowd‑pleaser |
| **Triple‑Layer Fruit Cake** | Three laye

In [5]:
system_prompt = """
ROLE:
You are a friendly customer support assistant for Sweet Crumbs Home Bakery.

FACTS:
- Plum Cake: Rs. 500
- Chocolate Cake: Rs. 650
- Cupcake: Rs. 80
- Eggless versions are available for all cakes.

RULES:
- Customers must order at least 24 hours in advance.
- Never invent menu items or prices.
- If information is unavailable, politely say you don't know.
- Be polite and helpful.

FORMAT:
- Use simple English.
- Keep responses short and clear.
- Use bullet points when listing products.
"""

print(system_prompt)


ROLE:
You are a friendly customer support assistant for Sweet Crumbs Home Bakery.

FACTS:
- Plum Cake: Rs. 500
- Chocolate Cake: Rs. 650
- Cupcake: Rs. 80
- Eggless versions are available for all cakes.

RULES:
- Customers must order at least 24 hours in advance.
- Never invent menu items or prices.
- If information is unavailable, politely say you don't know.
- Be polite and helpful.

FORMAT:
- Use simple English.
- Keep responses short and clear.
- Use bullet points when listing products.



In [6]:
response = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": "What cakes do you sell and what are their prices?"}
    ]
)
print(response.choices[0].message.content)

We sell:

- **Plum Cake** – ₹500  
- **Chocolate Cake** – ₹650  
- **Cupcake** – ₹80  

All of these have egg‑less versions available.


In [7]:
response = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": "Do you sell Black Forest Cake? What is its price?"}
    ]
)

print(response.choices[0].message.content)

I’m sorry, but we do not sell Black Forest Cake.  
Here are the cakes we do offer:

- Plum Cake – Rs. 500  
- Chocolate Cake – Rs. 650  
- Cupcake – Rs. 80  

All of these cakes come in eggless versions.  
If you’d like to order any of the above, please let us know at least 24 hours in advance.


### Observation – Messages, Roles and System Prompts

- System messages define the assistant's behaviour and instructions.
- User messages contain the customer's questions.
- Assistant messages contain the model's responses.
- Dividing a system prompt into ROLE, FACTS, RULES and FORMAT makes instructions easier to understand.
- Testing an unsupported cake request helps identify whether the assistant follows the given business information.

**Learning:** A well-designed system prompt helps guide the model, but its responses still need validation.

## 2. Few-Shot Prompting

Few-shot prompting means giving the language model a few examples so that it can understand the expected response pattern.

In this exercise, we provide sample customer questions and their corresponding assistant responses.

The model uses these examples as guidance when responding to a new customer question.

**Learning:** Examples can help the model produce more consistent responses.

In [8]:
messages = [
    {"role": "system", "content": system_prompt},

    {"role": "user", "content": "Do you have Plum Cake?"},
    {"role": "assistant", "content": "Yes! Plum Cake is available for Rs. 500."},

    {"role": "user", "content": "Is Chocolate Cake available?"},
    {"role": "assistant", "content": "Yes! Chocolate Cake costs Rs. 650."},

    {"role": "user", "content": "Do you have cupcakes?"}
]

response = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=messages
)

print(response.choices[0].message.content)

Yes, we have cupcakes.  

- Cupcake: Rs. 80  

Egg‑less versions are available.  
Please note you need to order at least 24 hours in advance.


## 3. JSON Output and Parsing

JSON (JavaScript Object Notation) is a structured format for storing and exchanging data.

We can instruct the language model to return customer or product information in JSON format.

In this exercise:

- The model generates a JSON response.
- `json.loads()` converts a JSON string into a Python dictionary.
- Individual values can be accessed using their keys.

**Learning:** Structured JSON responses make it easier for Python applications to process information.

In [9]:
import json

messages = [
    {
        "role": "system",
        "content": """
        You are a bakery assistant.
        Return the answer only in valid JSON.
        Use these keys:
        item, price, eggless_available.
        Do not include Markdown or extra text.
        """
    },
    {
        "role": "user",
        "content": "Tell me about Plum Cake."
    }
]

response = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=messages
)

result = response.choices[0].message.content

print(result)

{"item":"Plum Cake","price":"$9.50","eggless_available":true}


In [10]:
messages = [
    {
        "role": "system",
        "content": """
        You are a bakery assistant.

        Return only valid JSON with these keys:
        - item: string
        - price: integer in rupees
        - eggless_available: boolean

        Rules:
        - Plum Cake price is 500.
        - Do not use currency symbols.
        - Use true or false, not Yes or No.
        - Do not include Markdown or extra text.
        """
    },
    {
        "role": "user",
        "content": "Tell me about Plum Cake."
    }
]

response = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=messages
)

result = response.choices[0].message.content
print(result)

{"item":"Plum Cake","price":500,"eggless_available":false}


In [11]:
import json

result = '{"item":"Plum Cake","price":500,"eggless_available":true}'

data = json.loads(result)

print(data)
print(type(data))
print(data["item"])
print(data["price"])
print(data["eggless_available"])

{'item': 'Plum Cake', 'price': 500, 'eggless_available': True}
<class 'dict'>
Plum Cake
500
True


## 4. Python Dictionaries and JSON Serialization

A Python dictionary stores information using key-value pairs.

For our bakery, the menu dictionary stores cake names and their prices.

- `.get()` retrieves a value safely and can return a default when a key is missing.
- `json.dumps()` converts a Python dictionary into a JSON string.
- `json.loads()` converts a JSON string back into Python data.

**Learning:** Dictionaries and JSON help us store, retrieve and exchange structured information.

In [12]:
bakery_menu = {
    "plum cake": 500,
    "chocolate cake": 650,
    "cupcake": 80
}

print(bakery_menu.get("plum cake"))
print(bakery_menu.get("cupcake"))
print(bakery_menu.get("black forest cake", "Item not available"))

500
80
Item not available


In [13]:
import json

bakery_order = {
    "customer": "Alex",
    "item": "Chocolate Cake",
    "quantity": 2,
    "price": 650
}

json_data = json.dumps(bakery_order, indent=4)

print(json_data)
print(type(json_data))

{
    "customer": "Alex",
    "item": "Chocolate Cake",
    "quantity": 2,
    "price": 650
}
<class 'str'>


## 5. Python Functions, Type Hints and Docstrings

A function is a reusable block of code that performs a particular task.

In this exercise, `get_price()` calculates the total price of a bakery item based on its quantity.

- `item: str` indicates that the item is expected to be a string.
- `qty: int = 1` indicates that quantity is expected to be an integer, with a default value of 1.
- `-> int` indicates the expected return type.
- A docstring explains the purpose of the function.

**Learning:** Functions make code reusable, organized and easier to understand.

In [14]:
def get_price(item: str, qty: int = 1) -> int:
    """
    Calculate the total price of a bakery item.
    """
    bakery_menu = {
        "plum cake": 500,
        "chocolate cake": 650,
        "cupcake": 80
    }

    price = bakery_menu.get(item.lower())

    if price is None:
        return 0

    return price * qty


print(get_price("plum cake"))
print(get_price("chocolate cake", 2))
print(get_price("cupcake", 5))

500
1300
400


## 6. Python Decorators

A decorator is a function that adds extra functionality to another function without changing its original code.

In this exercise, we create a decorator called `log_call()`.

It displays:
- The name of the function being called.
- The result returned by the function.

We first apply it to a simple greeting function and then to our bakery price calculation function.

**Learning:** Decorators help add reusable functionality such as logging and monitoring.

In [15]:
def log_call(func):
    def wrapper(*args, **kwargs):
        print(f"Calling function: {func.__name__}")
        result = func(*args, **kwargs)
        print(f"Function returned: {result}")
        return result

    return wrapper


@log_call
def greet(name):
    return f"Hello, {name}!"


print(greet("Alex"))

Calling function: greet
Function returned: Hello, Alex!
Hello, Alex!


In [16]:
@log_call
def get_price(item: str, qty: int = 1) -> int:
    bakery_menu = {
        "plum cake": 500,
        "chocolate cake": 650,
        "cupcake": 80
    }

    price = bakery_menu.get(item.lower())

    if price is None:
        return 0

    return price * qty


total = get_price("chocolate cake", 2)
print("Total bill:", total)

Calling function: get_price
Function returned: 1300
Total bill: 1300


## 7. Exception Handling Using try and except

Exception handling allows a Python program to manage errors without stopping unexpectedly.

In this exercise, we learn two types of error handling:

1. Basic Python exceptions, such as `ValueError` and `ZeroDivisionError`.
2. API exceptions, such as `RateLimitError`, `APIConnectionError` and `APIStatusError`.

We use `try` to execute code that may fail and `except` to handle specific errors.

**Learning:** Exception handling makes applications more reliable and provides useful error messages to users.

In [17]:
try:
    number = int(input("Enter a number: "))
    result = 100 / number
    print("Result:", result)

except ValueError:
    print("Please enter a valid number.")

except ZeroDivisionError:
    print("Cannot divide by zero.")

Enter a number: 100
Result: 1.0


In [18]:
from openai import APIConnectionError, APIStatusError, RateLimitError

try:
    response = client.chat.completions.create(
        model="openai/gpt-oss-20b",
        messages=[
            {"role": "user", "content": "Suggest a cake for a birthday."}
        ]
    )

    print(response.choices[0].message.content)

except RateLimitError:
    print("API limit reached. Please try again later.")

except APIConnectionError:
    print("Unable to connect to Groq. Check your internet connection.")

except APIStatusError as e:
    print(f"Groq API error: {e.status_code}")

except Exception as e:
    print(f"Unexpected error: {e}")

Here are a few fun, crowd‑pleasing cake ideas that work for almost any birthday:

| Cake | Flavor Profile | Decoration Ideas | Great For |
|------|----------------|------------------|-----------|
| **Classic Red Velvet** | Rich chocolate with a subtle tang of cocoa, topped with cream‑cheese frosting | Frost the edges with a smooth white icing, add a drizzle of dark chocolate, and scatter a few shiny red velvet candies or a “Happy Birthday” topper | Anyone who loves a classic look and a bit of elegance |
| **Lemon Raspberry Cheesecake** | Creamy cheesecake base with a bright lemon zest swirl and fresh raspberries | Smooth the top with a glossy lemon glaze, garnish with fresh berries and a lemon twist, or a playful “Birthday Bubbles” topper | Perfect for a bright, refreshing celebration |
| **Chocolate‑Truffle Dream** | Deep, fudgy chocolate cake with a ganache layer | Use a silky ganache glaze, scatter chocolate shavings, and top with a gold‑leaf “Birthday” accent or a mini chocolate ba

In [19]:
response = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[
        {"role": "system", "content": system_prompt},
        {
            "role": "user",
            "content": "Suggest a birthday cake from your menu."
        }
    ]
)

print(response.choices[0].message.content)

For a birthday you can choose from our two main cakes:

- **Plum Cake – Rs. 500**  
- **Chocolate Cake – Rs. 650**  

Both have eggless versions if you prefer.  
Please remember to place your order at least 24 hours in advance.


## 8. Conversation Memory

A language model does not automatically remember previous messages between separate API calls.

We can maintain conversation memory using a Python list called `chat_history`.

Each message contains:
- A role (`system`, `user` or `assistant`).
- Its content.

Whenever a customer sends a message, we add it to the history. We also store the assistant's response.

By sending this history with the next API request, the model can use previous messages to answer follow-up questions.

**Learning:** Conversation history allows a chatbot to maintain context across multiple messages.

In [20]:
chat_history = [
    {"role": "system", "content": system_prompt}
]

def chat(user_message: str) -> str:
    chat_history.append(
        {"role": "user", "content": user_message}
    )

    try:
        response = client.chat.completions.create(
            model="openai/gpt-oss-20b",
            messages=chat_history
        )

        assistant_reply = response.choices[0].message.content

        chat_history.append(
            {"role": "assistant", "content": assistant_reply}
        )

        return assistant_reply

    except Exception as e:
        return f"Something went wrong: {e}"


print(chat("My name is Alex."))

Hi Alex! 👋

How can I help you today?  
If you’d like to place an order, please note:  
- Orders must be made at least 24 hours in advance.

Here’s our menu:

- **Plum Cake** – Rs. 500  
- **Chocolate Cake** – Rs. 650  
- **Cupcake** – Rs. 80  

All cakes are also available eggless. Let me know what you’d like to order!


In [21]:
print(chat("What is my name, and what bakery are we talking about?"))

Your name is Alex, and we are talking about Sweet Crumbs Home Bakery.


## 9. Memory Trimming

As a chatbot conversation grows, the number of messages sent to the language model also increases.

Sending too many messages can increase token usage.

Memory trimming helps control conversation history by retaining recent messages while removing older ones from the context.

In this exercise:
- `MAX_TURNS` defines the maximum number of conversation exchanges to retain.
- `trim_history()` preserves the system message.
- Recent user and assistant messages are retained.

**Learning:** Memory trimming helps manage conversation length and token usage, but older information may no longer be available to the model.

In [22]:
MAX_TURNS = 5

def trim_history():
    global chat_history

    system_message = chat_history[0]

    recent_messages = chat_history[1:][-(MAX_TURNS * 2):]

    chat_history = [system_message] + recent_messages


print("Messages before trimming:", len(chat_history))

trim_history()

print("Messages after trimming:", len(chat_history))

Messages before trimming: 5
Messages after trimming: 5


### Integrating Memory Trimming into the Chatbot

In this step, we combine conversation memory and memory trimming into the chatbot.

The updated `chat()` function:

1. Receives a customer message.
2. Adds the message to conversation history.
3. Trims older messages before sending the request.
4. Sends the available conversation history to Groq.
5. Stores the assistant's response.
6. Trims the history again to maintain the configured limit.

**Learning:** Integrating memory trimming into the chatbot helps control the conversation context automatically.

In [23]:
MAX_TURNS = 5

def trim_history():
    global chat_history

    system_message = chat_history[0]

    recent_messages = chat_history[1:][-(MAX_TURNS * 2):]

    chat_history = [system_message] + recent_messages


def chat(user_message: str) -> str:

    chat_history.append(
        {"role": "user", "content": user_message}
    )

    # Trim history before sending it to the model
    trim_history()

    try:
        response = client.chat.completions.create(
            model="openai/gpt-oss-20b",
            messages=chat_history
        )

        assistant_reply = response.choices[0].message.content

        chat_history.append(
            {"role": "assistant", "content": assistant_reply}
        )

        # Keep the history within the limit
        trim_history()

        return assistant_reply

    except Exception as e:
        return f"Something went wrong: {e}"


print(chat("What products do you sell?"))

We sell:

- Plum Cake – Rs. 500  
- Chocolate Cake – Rs. 650  
- Cupcake – Rs. 80  

All cakes come in eggless versions.  

Let me know if you want to order!


In [24]:
print("Total messages:", len(chat_history))
print("System role:", chat_history[0]["role"])
print("Latest message:", chat_history[-1]["content"])

Total messages: 7
System role: system
Latest message: We sell:

- Plum Cake – Rs. 500  
- Chocolate Cake – Rs. 650  
- Cupcake – Rs. 80  

All cakes come in eggless versions.  

Let me know if you want to order!


## 10. Final Interactive Chatbot

Finally, we create an interactive chatbot using the concepts learned throughout Day 2.

The customer can enter messages continuously, and the assistant responds using the Groq language model.

The chatbot uses:
- Structured system instructions.
- Python functions.
- Conversation memory.
- Memory trimming.
- API exception handling.

Type `exit` to end the conversation.

**Note:** This is a chatbot demonstration. It does not create actual bookings, process payments or save orders in a database.

In [25]:
print("🍰 Welcome to Sweet Crumbs Home Bakery!")
print("Type 'exit' to end the conversation.\n")

while True:
    user_input = input("You: ")

    if user_input.lower() == "exit":
        print("Bakery Assistant: Thank you for visiting Sweet Crumbs!")
        break

    reply = chat(user_input)

    print("Bakery Assistant:", reply)
    print()

🍰 Welcome to Sweet Crumbs Home Bakery!
Type 'exit' to end the conversation.

You: hi
Bakery Assistant: Hello! 👋  
How can I help you today?

You: I need birthday cake
Bakery Assistant: Sure! We have a few cake options that are great for birthdays.

- **Plum Cake** – Rs. 500  
- **Chocolate Cake** – Rs. 650  
- **Cupcake** – Rs. 80 (you can get a whole batch)

All of them are available eggless if you need.

To place an order:
- Let me know which cake you’d like.  
- Tell me the date you need it (must be at least 24 hours before delivery).  

Feel free to ask any other questions!

You: Suggest best cake for day  after tommorrow to feed 10 people
Bakery Assistant: For 10 people on the day after tomorrow, the best options are:

- **Plum Cake** – Rs. 500  
  * A rich, moist cake that’s great for a group.  
- **Chocolate Cake** – Rs. 650  
  * A classic favorite; very popular for parties.  

If you’d like, we can also send a batch of **Cupcakes** (Rs. 80 each). A dozen cupcakes would serve 1

# Day 2 – Observations, Learnings and Conclusion

## Topics Learned

1. Connected Python applications to Groq using an OpenAI-compatible API.
2. Understood system, user and assistant message roles.
3. Created structured system prompts using ROLE, FACTS, RULES and FORMAT.
4. Implemented few-shot prompting using example conversations.
5. Generated JSON responses and parsed them using `json.loads()`.
6. Worked with Python dictionaries, `.get()` and `json.dumps()`.
7. Created reusable functions with type hints and docstrings.
8. Implemented decorators for function logging.
9. Handled Python and API errors using `try/except`.
10. Implemented conversation memory using a Python list.
11. Added memory trimming using `MAX_TURNS`.
12. Built an interactive bakery chatbot using the Groq language model.

## Practical Observations

- Few-shot examples helped guide the model's response pattern.
- Structured system prompts helped restrict responses to the bakery menu.
- JSON made information easier to process using Python.
- Dictionaries provided a simple way to store menu items and prices.
- Decorators added logging functionality to existing functions.
- Exception handling helped manage API errors.
- Conversation history allowed the assistant to use previous messages.
- Memory trimming controlled the number of messages sent to the model.

## Limitations Discovered

During testing, I observed that:

- The chatbot sometimes generated unsupported delivery promises.
- It could incorrectly confirm orders without actual order processing.
- It could invent payment options.
- It does not have a database to save customer orders.
- Its memory depends on the conversation history maintained in Python.
- System prompts alone cannot guarantee that every response will follow business rules.

## Conclusion

Day 2 helped me understand how prompts, JSON, Python functions, decorators, exception handling and conversation memory work together in an AI chatbot.

I successfully developed an interactive Sweet Crumbs Home Bakery assistant using Python and the Groq API.

I also identified limitations related to hallucinations, order confirmation and the absence of database integration.

These concepts provide a foundation for developing more reliable AI agents in the upcoming workshop sessions.